# Open-source vision-language models vs CNNs for melanoma detection — experiment notebook

Group: Timothée Bruniaux, Karim Nasri, Anna Reedtz (UiT)

1. download HAM10000 and build a **lesion-level** split (training pool / 40 validation / 200 test images),
2. fine-tune **ResNet50** and **EfficientNet-B0**,
3. run two **open-source multimodal LLMs** locally on the Colab GPU (no API key, no cost):
   **Qwen2.5-VL-7B-Instruct** and **LLaVA-OneVision-7B**, zero-shot and few-shot,
4. compute sensitivity, specificity, AUC with bootstrap CIs, McNemar and DeLong tests, repeatability (SQ2) and the small-lesion subgroup (SQ3),
5. zip everything in `results/` for the report.

**Before you start:** `Runtime → Change runtime type → T4 GPU`. Everything is saved on Google Drive, so a cell can be re-run after a disconnection without losing work.

In [ ]:
!pip -q install timm statsmodels opencv-python-headless bitsandbytes accelerate qwen-vl-utils huggingface_hub
!pip -q install -U transformers

## 0. Configuration

In [ ]:
import os, json, time, hashlib, random, base64, io, re, glob, zipfile, datetime
import numpy as np, pandas as pd

from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/melanoma_llm_cnn'     # everything is saved here
DATA = f'{ROOT}/data'
RES  = f'{ROOT}/results'
os.makedirs(DATA, exist_ok=True); os.makedirs(RES, exist_ok=True)

SEED = 2026
N_TEST_PER_CLASS = 100       # 100 melanoma + 100 nevi
N_VAL_PER_CLASS  = 20        # 20 + 20
N_REPEATS        = 3         # LLM repeats per test image (SQ2)
N_BOOT           = 2000


random.seed(SEED); np.random.seed(SEED)
print('Results folder:', RES)

## 1. Download HAM10000
Harvard Dataverse blocks automated downloads from Colab (HTTP 403), so the same HAM10000 images and the official `HAM10000_metadata.csv` are taken from the public Hugging Face mirror `marmal88/skin_cancer`. Duplicated images across the mirror's splits are removed by `image_id`; the cell checks that 10,015 unique images are recovered.

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files
REPO = 'marmal88/skin_cancer'
IMG = '/content/ham_images'
os.makedirs(IMG, exist_ok=True)

meta_local = hf_hub_download(REPO, 'HAM10000_metadata.csv', repo_type='dataset', local_dir=DATA)
seen = set()
for f in sorted(x for x in list_repo_files(REPO, repo_type='dataset') if x.endswith('.parquet')):
    df = pd.read_parquet(hf_hub_download(REPO, f, repo_type='dataset'), columns=['image_id', 'image'])
    for iid, im in zip(df.image_id, df.image):
        if iid in seen: continue
        seen.add(iid)
        out = f'{IMG}/{iid}.jpg'
        if not os.path.exists(out):
            with open(out, 'wb') as fh: fh.write(im['bytes'])
    print(f, '->', len(seen), 'unique images so far')

img_path = {os.path.basename(p)[:-4]: p for p in glob.glob(f'{IMG}/*.jpg')}
meta_check = pd.read_csv(meta_local)
print(len(img_path), 'unique images found;', len(meta_check), 'rows in the official metadata')
missing = set(meta_check.image_id) - set(img_path)
print('Metadata images missing from the mirror:', len(missing))
assert len(missing) == 0, 'Some HAM10000 images are missing'
json.dump({'source': 'huggingface.co/datasets/' + REPO, 'n_images': len(img_path)}, open(f'{RES}/data_source.json', 'w'))

## 2. Lesion-level split (frozen before any model is run)

In [ ]:
meta_file = [p for p in glob.glob(f'{DATA}/HAM10000_metadata*')][0]
meta = pd.read_csv(meta_file, sep=None, engine='python')
meta = meta[meta.dx.isin(['mel', 'nv'])].copy()
meta = meta[meta.image_id.isin(img_path)]
meta['y'] = (meta.dx == 'mel').astype(int)
print(meta.dx.value_counts(), '\n', pd.crosstab(meta.dx, meta.dx_type))

split_file = f'{RES}/split.csv'
if os.path.exists(split_file):
    split = pd.read_csv(split_file); print('Existing frozen split loaded.')
else:
    rng = np.random.default_rng(SEED)
    les = meta.groupby('lesion_id').agg(dx=('dx', 'first'), histo=('dx_type', lambda s: (s == 'histo').all())).reset_index()
    test_l, val_l = [], []
    for dx in ['mel', 'nv']:
        pool = les[(les.dx == dx) & les.histo].lesion_id.values
        pick = rng.choice(pool, N_TEST_PER_CLASS + N_VAL_PER_CLASS, replace=False)
        test_l += list(pick[:N_TEST_PER_CLASS]); val_l += list(pick[N_TEST_PER_CLASS:])
    meta['set'] = 'train'
    meta.loc[meta.lesion_id.isin(val_l), 'set'] = 'val'
    meta.loc[meta.lesion_id.isin(test_l), 'set'] = 'test'
    # one image per lesion in val/test
    keep = meta[meta.set == 'train']
    one = meta[meta.set != 'train'].groupby('lesion_id', group_keys=False).apply(lambda g: g.sample(1, random_state=SEED))
    split = pd.concat([keep, one])[['image_id', 'lesion_id', 'dx', 'dx_type', 'age', 'sex', 'localization', 'y', 'set']]
    # internal hold-out (15 % of training lesions) for CNN early stopping
    tr_les = split[split.set == 'train'].lesion_id.unique()
    hold = rng.choice(tr_les, int(0.15 * len(tr_les)), replace=False)
    split.loc[split.lesion_id.isin(hold) & (split.set == 'train'), 'set'] = 'holdout'
    split.to_csv(split_file, index=False)

assert not set(split[split.set=='test'].lesion_id) & set(split[split.set!='test'].lesion_id), 'LEAKAGE'
test_ids = sorted(split[split.set == 'test'].image_id)
print(split.groupby(['set', 'dx']).size().unstack())
print('SHA-256 of frozen test list:', hashlib.sha256(','.join(test_ids).encode()).hexdigest())

## 3. Lesion area (proxy for lesion size, SQ3)
Uses the HAM10000 segmentation masks when they are in `data/`, otherwise Otsu thresholding.

In [ ]:
import cv2
seg = {}
for z in glob.glob(f'{DATA}/*segmentation*.zip'):
    with zipfile.ZipFile(z) as zf: zf.extractall('/content/ham_seg')
seg = {os.path.basename(p).split('_segmentation')[0]: p for p in glob.glob('/content/ham_seg/**/*.png', recursive=True)}

def lesion_area(iid):
    if iid in seg:
        m = cv2.imread(seg[iid], 0); return float((m > 127).mean()), 'mask'
    g = cv2.GaussianBlur(cv2.imread(img_path[iid], 0), (11, 11), 0)
    _, th = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(th)
    if n <= 1: return float('nan'), 'otsu'
    return float(stats[1:, cv2.CC_STAT_AREA].max() / th.size), 'otsu'

area = pd.DataFrame([(i, *lesion_area(i)) for i in test_ids], columns=['image_id', 'area_frac', 'area_method'])
area['size_tercile'] = pd.qcut(area.area_frac, 3, labels=['small', 'medium', 'large'])
area.to_csv(f'{RES}/test_lesion_area.csv', index=False); area.area_method.value_counts()

## 4. CNN baselines (ResNet50, EfficientNet-B0)

In [ ]:
import torch, timm, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image
from sklearn.metrics import roc_auc_score, roc_curve
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print(dev)
torch.manual_seed(SEED)

norm = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
tf_train = T.Compose([T.Resize((224, 224)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                      T.RandomRotation(20), T.ColorJitter(0.1, 0.1, 0.1), T.ToTensor(), norm])
tf_eval = T.Compose([T.Resize((224, 224)), T.ToTensor(), norm])

class DS(Dataset):
    def __init__(s, df, tf): s.df, s.tf = df.reset_index(drop=True), tf
    def __len__(s): return len(s.df)
    def __getitem__(s, i):
        r = s.df.iloc[i]; return s.tf(Image.open(img_path[r.image_id]).convert('RGB')), torch.tensor(float(r.y))

def predict(model, df):
    model.eval(); out = []
    with torch.no_grad(), torch.autocast(dev, enabled=dev == 'cuda'):
        for x, _ in DataLoader(DS(df, tf_eval), batch_size=64, num_workers=2):
            out.append(torch.sigmoid(model(x.to(dev)).float().squeeze(1)).cpu())
    return torch.cat(out).numpy()

def train_cnn(arch, max_epochs=15, patience=3):
    tr, ho = split[split.set == 'train'], split[split.set == 'holdout']
    model = timm.create_model(arch, pretrained=True, num_classes=1).to(dev)
    pos_w = torch.tensor((tr.y == 0).sum() / (tr.y == 1).sum(), device=dev)
    lossf = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler(enabled=dev == 'cuda')
    dl = DataLoader(DS(tr, tf_train), batch_size=32, shuffle=True, num_workers=2)
    best, bad, log = -1, 0, []
    for ep in range(max_epochs):
        model.train()
        for x, y in dl:
            opt.zero_grad()
            with torch.autocast(dev, enabled=dev == 'cuda'):
                loss = lossf(model(x.to(dev)).squeeze(1), y.to(dev))
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        auc = roc_auc_score(ho.y, predict(model, ho)); log.append({'arch': arch, 'epoch': ep + 1, 'holdout_auc': auc})
        print(arch, 'epoch', ep + 1, 'hold-out AUC', round(auc, 4))
        if auc > best: best, bad = auc, 0; torch.save(model.state_dict(), f'{ROOT}/{arch}.pt')
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(torch.load(f'{ROOT}/{arch}.pt'))
    return model, best, log

cnn_rows, train_log = [], []
val_df, test_df = split[split.set == 'val'], split[split.set == 'test']
CNN_DONE = os.path.exists(f'{RES}/cnn_predictions.csv')
if CNN_DONE: print('CNN predictions already on Drive -> training skipped.')
for arch in ([] if CNN_DONE else ['resnet50', 'efficientnet_b0']):
    m, best_auc, log = train_cnn(arch); train_log += log
    pv, pt = predict(m, val_df), predict(m, test_df)
    fpr, tpr, thr = roc_curve(val_df.y, pv); t = float(thr[np.argmax(tpr - fpr)])   # Youden on validation
    for split_name, df_, p in [('val', val_df, pv), ('test', test_df, pt)]:
        for iid, y, pp in zip(df_.image_id, df_.y, p):
            cnn_rows.append({'arm': arch, 'set': split_name, 'image_id': iid, 'y': y, 'score': float(pp),
                             'pred': int(pp >= t), 'threshold': t, 'holdout_auc': best_auc})
if not CNN_DONE:
    pd.DataFrame(cnn_rows).to_csv(f'{RES}/cnn_predictions.csv', index=False)
    pd.DataFrame(train_log).to_csv(f'{RES}/cnn_training_log.csv', index=False)
print('CNN done.')

## 5. LLM arms (Qwen2.5-VL-7B and LLaVA-OneVision-7B, zero-shot and few-shot)

**Main analysis (deterministic):** for each image we read the model's next-token log-probabilities after the prefix `Diagnosis:` and compare the first token of ` melanoma` with the first token of ` nevus`.
`p_melanoma = softmax(logp_melanoma, logp_nevus)` is used for the AUC, and the label is the more likely of the two (`p ≥ 0.5`). This is equivalent to greedy decoding restricted to the two answers, gives no invalid outputs, and is exactly reproducible.

**SQ2 (repeatability):** the zero-shot prompt is also answered 3 times with sampling (temperature 0.7), as a chat user would see it; we measure agreement between the 3 answers and the share of answers that are not a clear diagnosis.

Prompts may be changed only in the pilot / validation step (at most 3 revisions, noted in `PROMPT_LOG`).

In [ ]:
import gc, torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

LLMS = {'qwen': 'Qwen/Qwen2.5-VL-7B-Instruct',
        'llava': 'llava-hf/llava-onevision-qwen2-7b-ov-hf'}
# If the GPU runs out of memory, use the smaller versions instead:
# LLMS = {'qwen': 'Qwen/Qwen2.5-VL-3B-Instruct', 'llava': 'llava-hf/llava-onevision-qwen2-0.5b-ov-hf'}

PROMPT_VERSION = 'v2'
PROMPT_LOG = ['v1: initial prompt',
              'v2: same prompt; LLaVA-OneVision images resized to 384x384 because the few-shot prompt (5 images) ran out of GPU memory in v1. All arms re-run under v2.']
LLAVA_SIZE = 384
PROMPT = ('This is a dermoscopic image of one melanocytic skin lesion. '
          'Is it a melanoma or a benign nevus? '
          'Answer in exactly this form: "Diagnosis: melanoma" or "Diagnosis: nevus".')
PREFIX = 'Diagnosis:'
SAMPLE_TEMPERATURE = 0.7

# 4 fixed few-shot examples from the TRAINING pool (histology-confirmed)
rng_fs = np.random.default_rng(SEED)
tr_h = split[(split.set == 'train') & (split.dx_type == 'histo')]
FEWSHOT = [(i, 'melanoma') for i in rng_fs.choice(tr_h[tr_h.y == 1].image_id, 2, replace=False)] + \
          [(i, 'nevus') for i in rng_fs.choice(tr_h[tr_h.y == 0].image_id, 2, replace=False)]
rng_fs.shuffle(FEWSHOT)
json.dump(FEWSHOT, open(f'{RES}/fewshot_examples.json', 'w'))

def load_llm(name):
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
    kw = {'min_pixels': 128 * 28 * 28, 'max_pixels': 448 * 448} if name == 'qwen' else {}
    proc = AutoProcessor.from_pretrained(LLMS[name], **kw)
    model = AutoModelForImageTextToText.from_pretrained(LLMS[name], quantization_config=bnb,
                                                        device_map='cuda:0', torch_dtype=torch.float16)
    model.eval()
    tok = proc.tokenizer
    ids = (tok.encode(' melanoma', add_special_tokens=False)[0], tok.encode(' nevus', add_special_tokens=False)[0])
    assert ids[0] != ids[1], 'melanoma / nevus share their first token'
    print(name, 'loaded; answer token ids', ids, [tok.decode([i]) for i in ids])
    return proc, model, ids

def unload(*objs):
    for o in objs: del o
    gc.collect(); torch.cuda.empty_cache()

torch.cuda.empty_cache()

def pil(iid): return Image.open(img_path[iid]).convert('RGB')

def build(proc, iid, few):
    small = 'llava' in type(proc).__name__.lower()
    load = (lambda i: pil(i).resize((LLAVA_SIZE, LLAVA_SIZE))) if small else pil
    content, images = [], []
    if few:
        for k, (e, lab) in enumerate(FEWSHOT, 1):
            content += [{'type': 'text', 'text': f'Example {k}:'}, {'type': 'image'},
                        {'type': 'text', 'text': f'Diagnosis: {lab}'}]
            images.append(load(e))
        content.append({'type': 'text', 'text': 'Now the image to classify:'})
    content += [{'type': 'image'}, {'type': 'text', 'text': PROMPT}]
    images.append(load(iid))
    text = proc.apply_chat_template([{'role': 'user', 'content': content}], add_generation_prompt=True)
    return text, images

@torch.no_grad()
def score(proc, model, ids, iid, few):
    text, images = build(proc, iid, few)
    inp = proc(text=[text + PREFIX], images=images, return_tensors='pt').to(model.device)
    lp = torch.log_softmax(model(**inp).logits[0, -1].float(), -1)
    lm, ln = lp[ids[0]].item(), lp[ids[1]].item()
    return 1 / (1 + np.exp(ln - lm)), lm, ln

@torch.no_grad()
def sample(proc, model, iid):
    text, images = build(proc, iid, False)
    inp = proc(text=[text], images=images, return_tensors='pt').to(model.device)
    out = model.generate(**inp, do_sample=True, temperature=SAMPLE_TEMPERATURE, top_p=1.0, max_new_tokens=40)
    return proc.batch_decode(out[:, inp['input_ids'].shape[1]:], skip_special_tokens=True)[0]

def parse(txt):
    m = re.search(r'\b(melanoma|nevus|naevus)\b', (txt or '').lower())
    return {'naevus': 'nevus'}.get(m.group(1), m.group(1)) if m else None

SCORES, SAMPLES = f'{RES}/llm_scores.csv', f'{RES}/llm_samples.csv'
def done(path, cols):
    if not os.path.exists(path): return set()
    return set(map(tuple, pd.read_csv(path)[cols].astype(str).values))

def append(path, rec):
    pd.DataFrame([rec]).to_csv(path, mode='a', header=not os.path.exists(path), index=False)

truth = split.set_index('image_id').y
def run_scores(name, proc, model, ids, set_name, image_ids):
    d = done(SCORES, ['arm', 'set', 'image_id', 'prompt_version'])
    for few in (False, True):
        arm = f"{name}_{'few' if few else 'zero'}"
        for iid in image_ids:
            if (arm, set_name, iid, PROMPT_VERSION) in d: continue
            t0 = time.time(); p, lm, ln = score(proc, model, ids, iid, few)
            append(SCORES, {'arm': arm, 'set': set_name, 'image_id': iid, 'prompt_version': PROMPT_VERSION,
                            'y': int(truth[iid]), 'p_melanoma': p, 'logp_melanoma': lm, 'logp_nevus': ln,
                            'model': LLMS[name], 'seconds': round(time.time() - t0, 2)})
        print(arm, set_name, 'done')

def run_samples(name, proc, model, set_name, image_ids, repeats=3):
    d = done(SAMPLES, ['arm', 'set', 'image_id', 'repeat', 'prompt_version'])
    arm = f'{name}_zero'
    for rep in range(repeats):
        torch.manual_seed(SEED + rep)
        for iid in image_ids:
            if (arm, set_name, iid, str(rep), PROMPT_VERSION) in d: continue
            txt = sample(proc, model, iid)
            append(SAMPLES, {'arm': arm, 'set': set_name, 'image_id': iid, 'repeat': rep, 'prompt_version': PROMPT_VERSION,
                             'y': int(truth[iid]), 'raw': txt, 'diagnosis': parse(txt), 'model': LLMS[name]})
        print(arm, 'sampled repeat', rep, 'done')

### 5a. Pilot + validation (40 validation images). Check the AUC and that the answer tokens are right. If the validation AUC is near 0.5 for a model, revise the prompt (new `PROMPT_VERSION`, add a line to `PROMPT_LOG`) and re-run. Max 3 revisions.

In [ ]:
from sklearn.metrics import roc_auc_score
val_ids = list(val_df.image_id)
for name in LLMS:
    proc, model, ids = load_llm(name)
    print('Example answer:', sample(proc, model, val_ids[0]))
    run_scores(name, proc, model, ids, 'val', val_ids)
    del model, proc; gc.collect(); torch.cuda.empty_cache()
s = pd.read_csv(SCORES); v = s[(s.set == 'val') & (s.prompt_version == PROMPT_VERSION)]
print(v.groupby('arm').apply(lambda g: pd.Series({'auc': roc_auc_score(g.y, g.p_melanoma),
                                                  'acc': ((g.p_melanoma >= .5) == g.y).mean(),
                                                  'sec_per_image': g.seconds.mean()})))
json.dump({'final_prompt_version': PROMPT_VERSION, 'prompt': PROMPT, 'prefix': PREFIX, 'log': PROMPT_LOG,
           'models': LLMS}, open(f'{RES}/prompt_final.json', 'w'), indent=1)

### 5b. TEST RUN — 200 images × 2 models × (zero-shot + few-shot) scoring, then 3 sampled answers per image (zero-shot). Prompts are now frozen. About 2–3 hours; if Colab disconnects, re-run the cell and it continues.

In [ ]:
for name in LLMS:
    proc, model, ids = load_llm(name)
    run_scores(name, proc, model, ids, 'test', test_ids)
    run_samples(name, proc, model, 'test', test_ids, repeats=N_REPEATS)
    del model, proc; gc.collect(); torch.cuda.empty_cache()
print('Test run finished:', datetime.datetime.utcnow().isoformat())

## 6. Analysis

In [ ]:
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters
from sklearn.metrics import f1_score, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

ARMS = ['qwen_zero', 'qwen_few', 'llava_zero', 'llava_few']
sc = pd.read_csv(SCORES); sc = sc[(sc.set == 'test') & (sc.prompt_version == PROMPT_VERSION)]
cnn = pd.read_csv(f'{RES}/cnn_predictions.csv'); cnn = cnn[cnn.set == 'test']

def llm_table(arm):
    d = sc[sc.arm == arm].set_index('image_id').loc[test_ids]
    return pd.DataFrame({'image_id': d.index, 'y': d.y.values, 'pred': (d.p_melanoma >= 0.5).astype(int).values,
                         'score': d.p_melanoma.values})

tables = {a: cnn[cnn.arm == a].set_index('image_id').loc[test_ids].reset_index()[['image_id', 'y', 'pred', 'score']]
          for a in ['resnet50', 'efficientnet_b0']}
tables.update({a: llm_table(a) for a in ARMS})

def metr(y, pred, score):
    y, pred = np.asarray(y), np.asarray(pred)
    return {'sensitivity': pred[y == 1].mean(), 'specificity': 1 - pred[y == 0].mean(),
            'accuracy': (pred == y).mean(), 'f1': f1_score(y, pred), 'auc': roc_auc_score(y, score)}

def boot_ci(t, n=N_BOOT):
    rng = np.random.default_rng(SEED); pos, neg = np.where(t.y == 1)[0], np.where(t.y == 0)[0]; res = []
    for _ in range(n):
        i = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        res.append(metr(t.y.values[i], t.pred.values[i], t.score.values[i]))
    return pd.DataFrame(res).quantile([0.025, 0.975])

rows = []
for a, t in tables.items():
    m = metr(t.y, t.pred, t.score); ci = boot_ci(t)
    rows.append({'arm': a, **{k: round(v, 3) for k, v in m.items()},
                 **{f'{k}_ci': f'{ci[k].iloc[0]:.3f}-{ci[k].iloc[1]:.3f}' for k in m}})
metrics = pd.DataFrame(rows); metrics.to_csv(f'{RES}/metrics_main.csv', index=False); metrics

In [ ]:
# DeLong test (fast implementation, Sun & Xu 2014)
def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T_ = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]: j += 1
        T_[i:j] = 0.5 * (i + j - 1) + 1; i = j
    out = np.empty(N); out[J] = T_; return out

def delong(y, s1, s2):
    y = np.asarray(y); order = np.argsort(-y, kind='stable'); m = int(y.sum()); n = len(y) - m
    P = np.vstack([s1, s2])[:, order]
    tx = np.array([_midrank(p[:m]) for p in P]); ty = np.array([_midrank(p[m:]) for p in P]); tz = np.array([_midrank(p) for p in P])
    aucs = tz[:, :m].sum(1) / m / n - (m + 1) / 2 / n
    v01 = (tz[:, :m] - tx) / n; v10 = 1 - (tz[:, m:] - ty) / m
    cov = np.cov(v01) / m + np.cov(v10) / n
    z = (aucs[0] - aucs[1]) / np.sqrt(cov[0, 0] + cov[1, 1] - 2 * cov[0, 1])
    return aucs, 2 * stats.norm.sf(abs(z))

def paired_mcnemar(t1, t2, cls):
    s = t1.y == cls; c1 = (t1.pred == t1.y)[s].values; c2 = (t2.pred == t2.y)[s].values
    tab = [[(c1 & c2).sum(), (c1 & ~c2).sum()], [(~c1 & c2).sum(), (~c1 & ~c2).sum()]]
    return mcnemar(tab, exact=True).pvalue

def boot_auc_diff(t1, t2, n=N_BOOT):
    rng = np.random.default_rng(SEED); pos, neg = np.where(t1.y == 1)[0], np.where(t1.y == 0)[0]; d = []
    for _ in range(n):
        i = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        d.append(roc_auc_score(t1.y.values[i], t1.score.values[i]) - roc_auc_score(t2.y.values[i], t2.score.values[i]))
    return np.quantile(d, [0.025, 0.975])

# best CNN chosen on the internal hold-out (not on the test set)
best_cnn = cnn.groupby('arm').holdout_auc.first().idxmax(); print('Best CNN (hold-out):', best_cnn)
N_TESTS = 4 * 3
comp = []
for a in ARMS:
    t1, t2 = tables[a], tables[best_cnn]
    aucs, p_auc = delong(t1.y, t1.score, t2.score); lo, hi = boot_auc_diff(t1, t2)
    diff = aucs[0] - aucs[1]
    comp.append({'llm_arm': a, 'vs': best_cnn, 'auc_llm': aucs[0], 'auc_cnn': aucs[1], 'auc_diff': diff,
                 'auc_diff_ci': f'{lo:.3f} to {hi:.3f}', 'p_delong': p_auc, 'p_mcnemar_sens': paired_mcnemar(t1, t2, 1),
                 'p_mcnemar_spec': paired_mcnemar(t1, t2, 0),
                 'comparable(|diff|<0.05 & CI incl. 0)': abs(diff) < 0.05 and lo <= 0 <= hi})
comp = pd.DataFrame(comp)
for c in ['p_delong', 'p_mcnemar_sens', 'p_mcnemar_spec']: comp[c + '_bonf'] = np.minimum(comp[c] * N_TESTS, 1)
comp.round(4).to_csv(f'{RES}/comparisons_vs_best_cnn.csv', index=False); comp.round(4)

In [ ]:
# SQ1: zero-shot vs few-shot
sq1 = []
for llm in ['qwen', 'llava']:
    z, f = tables[f'{llm}_zero'], tables[f'{llm}_few']; aucs, p = delong(z.y, z.score, f.score)
    sq1.append({'llm': llm, 'auc_zero': aucs[0], 'auc_few': aucs[1], 'p_delong': p,
                'p_mcnemar_sens': paired_mcnemar(z, f, 1), 'p_mcnemar_spec': paired_mcnemar(z, f, 0)})
pd.DataFrame(sq1).round(4).to_csv(f'{RES}/sq1_zero_vs_few.csv', index=False)

# SQ2: repeatability of sampled answers (3 repeats, zero-shot prompt)
smp = pd.read_csv(SAMPLES); smp = smp[(smp.set == 'test') & (smp.prompt_version == PROMPT_VERSION)]
smp['diagnosis'] = smp.diagnosis.fillna('unclear')
sq2 = []
for a in ['qwen_zero', 'llava_zero']:
    lab = smp[smp.arm == a].pivot_table(index='image_id', columns='repeat', values='diagnosis', aggfunc='first')
    agg, _ = aggregate_raters(lab.values)
    greedy = tables[a].set_index('image_id').pred.map({1: 'melanoma', 0: 'nevus'})
    sq2.append({'arm': a, 'fleiss_kappa': fleiss_kappa(agg), 'identical_all_repeats': (lab.nunique(axis=1) == 1).mean(),
                'unclear_answer_rate': (smp[smp.arm == a].diagnosis == 'unclear').mean(),
                'agreement_with_main_label': (lab.eq(greedy.loc[lab.index], axis=0)).values.mean(),
                'sampled_accuracy': (smp[smp.arm == a].diagnosis == smp[smp.arm == a].y.map({1: 'melanoma', 0: 'nevus'})).mean()})
pd.DataFrame(sq2).round(4).to_csv(f'{RES}/sq2_repeatability.csv', index=False)

# SQ3: performance by lesion-size tercile
ar = pd.read_csv(f'{RES}/test_lesion_area.csv')
sq3 = []
for a, t in tables.items():
    tt = t.merge(ar, on='image_id')
    for ter, g in tt.groupby('size_tercile'):
        sq3.append({'arm': a, 'tercile': ter, 'n_mel': int(g.y.sum()), 'n_nv': int((g.y == 0).sum()),
                    'sensitivity': g.pred[g.y == 1].mean(), 'specificity': 1 - g.pred[g.y == 0].mean()})
pd.DataFrame(sq3).round(3).to_csv(f'{RES}/sq3_by_lesion_size.csv', index=False)

# Confusion matrices and run time
cm = [{'arm': a, 'TP': int(((t.pred == 1) & (t.y == 1)).sum()), 'FN': int(((t.pred == 0) & (t.y == 1)).sum()),
       'TN': int(((t.pred == 0) & (t.y == 0)).sum()), 'FP': int(((t.pred == 1) & (t.y == 0)).sum())} for a, t in tables.items()]
pd.DataFrame(cm).to_csv(f'{RES}/confusion_matrices.csv', index=False)
sc.groupby('arm').agg(images=('image_id', 'size'), mean_seconds=('seconds', 'mean'), model=('model', 'first')).to_csv(f'{RES}/llm_usage.csv')
print(pd.DataFrame(sq2).round(3)); print(pd.DataFrame(cm))

In [ ]:
# Figures
names = {'resnet50': 'ResNet50', 'efficientnet_b0': 'EfficientNet-B0', 'qwen_zero': 'Qwen2.5-VL zero-shot',
         'qwen_few': 'Qwen2.5-VL few-shot', 'llava_zero': 'LLaVA-OneVision zero-shot', 'llava_few': 'LLaVA-OneVision few-shot'}
plt.figure(figsize=(6, 6))
for a, t in tables.items():
    f, tp, _ = roc_curve(t.y, t.score); plt.plot(f, tp, label=f"{names[a]} (AUC {roc_auc_score(t.y, t.score):.2f})")
plt.plot([0, 1], [0, 1], 'k--', lw=0.8); plt.xlabel('1 - specificity'); plt.ylabel('Sensitivity'); plt.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{RES}/fig_roc.png', dpi=200); plt.show()

In [ ]:
# Environment info + zip of the results folder (send results.zip to the report writer)
import platform, sklearn, statsmodels, transformers
json.dump({'python': platform.python_version(), 'torch': torch.__version__, 'timm': timm.__version__,
           'transformers': transformers.__version__, 'sklearn': sklearn.__version__, 'statsmodels': statsmodels.__version__,
           'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu', 'seed': SEED, 'llms': LLMS,
           'finished_utc': datetime.datetime.utcnow().isoformat()}, open(f'{RES}/environment.json', 'w'), indent=1)
import shutil; shutil.make_archive(f'{ROOT}/results', 'zip', RES)
print('Download:', f'{ROOT}/results.zip')